In [139]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

THRESHOLD_TIMESTAMPS = 16
L_SEQUENCE_LENGHT = 48

In [140]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))


<class 'generator'>


In [141]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [142]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [143]:
sessions_in_dataset = OttoDataSetSession(sessions_bf_threshold)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght_after_threshold = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]["timestamps"]
    if len(sample) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(len(sample))

print(np.mean(session_sample_lenght_after_threshold))
print(f"The total of the median is for this total of {len(session_sample_lenght_after_threshold)} {np.median(session_sample_lenght_after_threshold)}")


Total len of the Sessions: 201
124.75324675324676
The total of the median is for this total of 154 104.5


In [146]:
class OttoDataSetSequenceLenght(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        self.session = session
    
    def __sequenceLenght__(self): 
        self.session = [i if i >= L_SEQUENCE_LENGHT else "add_padding" for i in self.session]
        return self.session

In [147]:
data_set_after_L = OttoDataSetSequenceLenght(session_sample_lenght_after_threshold)
print(data_set_after_L.__sequenceLenght__())

[276, 'add_padding', 'add_padding', 226, 'add_padding', 204, 'add_padding', 93, 81, 382, 86, 91, 'add_padding', 112, 372, 108, 108, 261, 'add_padding', 292, 76, 'add_padding', 126, 88, 115, 231, 213, 267, 151, 185, 310, 123, 129, 151, 'add_padding', 63, 376, 230, 188, 'add_padding', 235, 112, 'add_padding', 348, 149, 'add_padding', 'add_padding', 'add_padding', 'add_padding', 127, 'add_padding', 'add_padding', 59, 'add_padding', 56, 50, 53, 93, 'add_padding', 283, 141, 'add_padding', 'add_padding', 164, 272, 143, 192, 163, 'add_padding', 'add_padding', 61, 82, 443, 79, 157, 68, 'add_padding', 326, 120, 58, 'add_padding', 57, 'add_padding', 209, 256, 110, 172, 57, 'add_padding', 205, 'add_padding', 81, 'add_padding', 'add_padding', 'add_padding', 'add_padding', 110, 82, 59, 'add_padding', 135, 112, 'add_padding', 'add_padding', 234, 167, 72, 105, 'add_padding', 273, 'add_padding', 'add_padding', 265, 116, 187, 266, 63, 'add_padding', 217, 109, 241, 125, 'add_padding', 'add_padding', 102